## Online Retail Transactions

**Understanding the Dataset**

The online retail csv file is an extensive collection of data relating to ecommerce transactions. This dataset provides a detailed view of sales activities within the online retail sector, covering numerous essential attributes necessary for a quantitative understanding of consumer behavior and the overall business performance.

**The Columns**:
* **InvoiceNo**	A unique identification number assigned to each transaction. (Numeric)
* **StockCode**	A unique identification code assigned to each product sold by the retailer. (Numeric)
* **Description**	A brief description of the product sold. (Text)
* **Quantity**	The number of units of the product sold in each transaction. (Numeric)
* **InvoiceDate**	The exact date and time when the transaction occurred. (Date/Time)
* **UnitPrice**	The price per unit of the product sold. (Numeric)
* **Country**	The country where the customer resides. (Text)

Source: https://www.kaggle.com/datasets/thedevastator/online-retail-transaction-records

**Import Pyspark and start a SparkSession**

In [40]:
from pyspark.sql import SparkSession

In [41]:
# start a spark session
spark = (SparkSession.builder
         .master('local')
         .appName("online_retail_project")
         .getOrCreate())

# create a spark context
sc = spark.sparkContext

**Step 1: Load the csv file** 

In [13]:
# load data
data = sc.textFile('dataset/retail_dataset.csv')

In [43]:
# let's re-read the dataset using spark's csv reader to make sure all columns are correctly infered.
df = spark.read.csv('dataset/retail_dataset.csv', header=True, inferSchema=True)
data = df.rdd
print(type(data))

<class 'pyspark.rdd.RDD'>


confirmed that the data is of type rdd, that means our data was read successfully.

**Step 2: Cleaning Data**

Now let's get a sense of our dataset and possible data quality issues. Now, our RDD is full of a collection of row objects

**1. Row count**

In [47]:
# check count
print(f'There are {data.count()} of data')

There are 541909 of data


**2. Check columns**

In [49]:
# let's check the count of missing or null values in each column
missing_counts = data.map(lambda row: [(col, 1) if row[col] is None else (col, 0) for col in row.asDict()]) \
    .flatMap(lambda x: x) \
    .reduceByKey(lambda a, b: a + b) \
    .collect()

so we create a tuple containing the column and the number of rows, and we convert it as a dictionary so we can iterate over it, after we flatten each tuple, for for each flattened tuple we perform an aggreagation using reducebykey.

In [52]:
for col in missing_counts:
    print(col)

('index', 0)
('InvoiceNo', 0)
('StockCode', 0)
('Description', 1454)
('Quantity', 0)
('InvoiceDate', 0)
('UnitPrice', 0)
('CustomerID', 135080)
('Country', 0)


* We can see that there are 1454 missing values in the Description.
* 135k rows of data also have missing customer ids

**3. Preview data** - Check data distribution

In [61]:
# show a sample of the description column
data.map(lambda row: (row['Description'], 1)).reduceByKey(lambda a, b: a + b).take(10)

[('WHITE HANGING HEART T-LIGHT HOLDER', 2369),
 ('WHITE METAL LANTERN', 328),
 ('CREAM CUPID HEARTS COAT HANGER', 293),
 ('KNITTED UNION FLAG HOT WATER BOTTLE', 473),
 ('RED WOOLLY HOTTIE WHITE HEART.', 449),
 ('SET 7 BABUSHKA NESTING BOXES', 389),
 ('GLASS STAR FROSTED T-LIGHT HOLDER', 141),
 ('HAND WARMER UNION JACK', 515),
 ('HAND WARMER RED POLKA DOT', 18),
 ('ASSORTED COLOUR BIRD ORNAMENT', 1501)]

In [60]:
# preview the customer id column
data.map(lambda row: (row['CustomerID'], 1)).reduceByKey(lambda a, b: a + b).take(10)

[(17850.0, 312),
 (13047.0, 196),
 (12583.0, 251),
 (13748.0, 28),
 (15100.0, 6),
 (15291.0, 109),
 (14688.0, 359),
 (17809.0, 64),
 (15311.0, 2491),
 (14527.0, 1011)]

In [62]:
# preview Country data. 
data.map(lambda row: (row['Country'], 1)).reduceByKey(lambda a, b: a + b).take(10)

[('United Kingdom', 495478),
 ('France', 8557),
 ('Australia', 1259),
 ('Netherlands', 2371),
 ('Germany', 9495),
 ('Norway', 1086),
 ('EIRE', 8196),
 ('Switzerland', 2002),
 ('Spain', 2533),
 ('Poland', 341)]

In [64]:
# view missing values in CustomerID column
data.filter(lambda row: row['CustomerID'] is None).take(10)

[Row(index=622, InvoiceNo='536414', StockCode='22139', Description=None, Quantity=56, InvoiceDate='12/1/2010 11:52', UnitPrice=0.0, CustomerID=None, Country='United Kingdom'),
 Row(index=1443, InvoiceNo='536544', StockCode='21773', Description='DECORATIVE ROSE BATHROOM BOTTLE', Quantity=1, InvoiceDate='12/1/2010 14:32', UnitPrice=2.51, CustomerID=None, Country='United Kingdom'),
 Row(index=1444, InvoiceNo='536544', StockCode='21774', Description='DECORATIVE CATS BATHROOM BOTTLE', Quantity=2, InvoiceDate='12/1/2010 14:32', UnitPrice=2.51, CustomerID=None, Country='United Kingdom'),
 Row(index=1445, InvoiceNo='536544', StockCode='21786', Description='POLKADOT RAIN HAT ', Quantity=4, InvoiceDate='12/1/2010 14:32', UnitPrice=0.85, CustomerID=None, Country='United Kingdom'),
 Row(index=1446, InvoiceNo='536544', StockCode='21787', Description='RAIN PONCHO RETROSPOT', Quantity=2, InvoiceDate='12/1/2010 14:32', UnitPrice=1.66, CustomerID=None, Country='United Kingdom'),
 Row(index=1447, Invoic

**4. Replace missing values**

From the exploration, we can see that the required columns for KPI analysis (sales) is filled, and there are no nulls. Therefore dropping **null** customerIDs or Descriptions would remove these vital data points.

In [69]:
# replace all missing values in customer id

# because we created the dataframe before converting to rdd. the elements are spark row objects, and executors need to know what Row is when creating or modifying them. Without the import, they throw a NameError
from pyspark.sql import Row 

# extract the columns... convert the rdd to df to extract columns
columns = data.toDF().columns

updated_data = data.map(lambda row: Row(**
                                       {
                                           col: ("Unknown" if col == "CustomerID" and row[col] is None else row[col])
                                           for col in columns
                                       }))

In [72]:
updated_data.filter(lambda row: row['CustomerID'] is None).take(10)

[]

There are no nulls in the column now

**Step 3**

In [32]:
print("Number of columns are: ", len(header.split(",")))

Number of columns are:  9


From the column counts:
* 537k rows have 9 columns which is the original number of columns
* 3729 rows have 10 columns which means there are extra commas
* 1067 rows have 11 columns, just like the above. 

With the extra commas, there is proof that the data is malformed. We will inspect the malformed rows

In [34]:
## identify missing values per column

num_cols = len(header.split(','))

for i in range(num_cols):
    missing_count = split_data.filter(lambda row: row[i].strip() == "").count()
    print(f'Column {i} missing values: {missing_count}')

Column 0 missing values: 0
Column 1 missing values: 0
Column 2 missing values: 0
Column 3 missing values: 1454
Column 4 missing values: 0
Column 5 missing values: 0
Column 6 missing values: 0
Column 7 missing values: 133391
Column 8 missing values: 1365


columns 3 (Quantity), 7 (UnitPrice), 8 (Country) all have missing values. 

unit price columns that are missing will be dropped. But first let's double check the rows with more than 9 columns, some data might have been extended into another wrong column.